# 01 · Suppression-mechanism confirmation  —  `DIFF@pivot = ATTN(q = −PROGRESS, k = TASK)`

Hypothesis: at the pivot the goal heads compute the leftover task by a single attention op whose
**query is a function of PROGRESS** (the done-set) and whose **keys/values are the TASK spec**. The
query *suppresses* attention onto the task items already satisfied by PROGRESS; the surviving mass is
the frontier item, written by the OV.

Battery (one 3-predicate example, 2–3 instances — flow-polishing only, no aggregate stats):
- **A1** graded suppression + OV-write norm — does attention onto item-k collapse exactly when PROGRESS completes k?
- **A2** anti-alignment — is the query update from completing item i anti-parallel to item-i's task keys?
- **A3** sign necessity — reflecting the done-update through baseline should *not* skip to the frontier.
- **B1** query linearity (geometric) — `Δq({a,b}) ≈ Δq({a}) + Δq({b})`?
- **B2** compositional causal injection — inject `q({a})+q({b})−q({})`; does it *generate* item c only? saturation → boundary.

### 🔧 Intervention mechanism (reference) — pre-RoPE query transplant

*The same machinery underlies Spec 1 (`00_diff.ipynb`), A3, and B2. Read this once; every
`inject`-style arm below is a variation on it.* Numbers below are for **Qwen2.5-Coder-32B**
(40 q-heads, 8 kv-heads, `head_dim=128` → `q_proj` output width = 40×128 = 5120, `GROUP = 40/8 = 5`).

#### 1 · Source / destination prompts + pivots

Both prompts are the **same TASK**, differing only in how much of the prefill (**PROGRESS**) is
written. The task description precedes the prefill, so its tokens sit at **identical positions** in both
→ their **keys/values match and are never touched**. Only the **pivot query** differs.

```
                    ┌────────────── SHARED: TASK spec  (KEYS / VALUES live here) ──────────────┐
 SOURCE (pab)       │ Write func(words) that computes:                                          │
 "a,b done"         │  (1) `a`: words longer than 3 chars                                        │
                    │  (2) `b`: count of palindromic words                                       │
                    │  (3) `c`: length of the longest word                                       │
                    └───────────────────────────────────────────────────────────────────────────┘
                    <code>
                    def func(words: list[str]):
                        a = [w for w in words if len(w) > 3]   ┐ PROGRESS = {a,b}
                        b = sum(w == w[::-1] for w in words)   ┘ (this region DIFFERS)
                        ‸   ← PIVOT_SRC   (trailing indent = last token; about to write next var)

 DEST (p0)          ┌────────────── SAME TASK, SAME TOKEN POSITIONS ───────────────────────────┐
 "nothing done"     │ ... identical (1)(2)(3) ...                                                │
                    └───────────────────────────────────────────────────────────────────────────┘
                    <code>
                    def func(words: list[str]):
                        ‸   ← PIVOT_DEST   PROGRESS = {}
```

- **Pivot = last token of the prompt** (`pivot_of(p) = input_ids.shape[1] - 1`).
- SRC's pivot has a *larger absolute index* (extra a,b lines) — harmless: the query is cached
  **pre-RoPE** (position-independent) and re-rotated at DEST's own position on injection.

#### 2 · What is cached from the SRC pivot

```python
qab = cache_src_proj(pab, [pivot_of(pab)], Q_LAYERS, "q")   # {layer: (1, 5120)}
```

- Traces each goal-layer's `…self_attn.q_proj` **output** at the **pivot position only**.
- It is the **`q_proj` output, PRE-RoPE** (RoPE is applied later, inside attention).
- The full 5120-wide row is stored, but **only the goal-head 128-wide slices are used** on injection.
- `q_reflect` has no source prompt — it is *synthesized*: `q_lincomb([(2.0, q0), (-1.0, qab)])` = `2·q0 − qab`.

#### 3 · The injection — overwrite only goal-head q-slices at the pivot

`proj_patch_hooks` registers a forward hook on each `q_proj`; for each goal head `h` and target
position `p` it does `o[:, p, h*128:(h+1)*128] = qab[layer][row, h*128:(h+1)*128]`. All other heads
keep DEST's own query; **keys/values (TASK) are untouched**. RoPE rotates the injected vector at `p`.

```
   DEST q_proj output @ PIVOT_DEST, layer L:   [ h0 | h1 |███| h3 | … |███| … | h39 ]
                                                          ▲            ▲
                                                    goal heads overwritten with qab's slices
                                                    (███), everything else = DEST's own query
```

**Two phases (`greedy`):**

```
   PREFILL  ...task...  def func(...):  [PIVOT_DEST]‸    ← prefill_specs (positions=[PIVOT_DEST])
            → the pivot's patched attention leaves a modified residual + K/V in the KV cache
              (this one-shot footprint "rides the cache"; the QUERY itself is never cached)

   DECODE 1     [tok₁]‸    ← decode_specs (positions=[-1]): patch the NEW token's own query
   DECODE 2     [tok₂]‸    ← same fixed qab again  (queries are recomputed fresh every step)
   ...                        repeated for the first n_steer steps, then released
```

- **Queries are never stored in the KV cache — only K and V are.** So the prefill patch's persistence
  is only the pivot's *downstream K/V footprint*; to actually **hold** the steer you must re-inject the
  query every decode step (`decode_specs`, `positions=[-1]`).
- Each step stamps the **same fixed pre-RoPE `qab`** onto an ordinary (non-pivot) generated token,
  overwriting the query it would have computed. This **freezes PROGRESS** → run forever it would loop;
  `n_steer` caps it (e.g. 20) so the frontier decision carries through without runaway.

> **vs Spec 1 (`00_diff`):** identical mechanism (pre-RoPE `q_proj`-output transplant, re-clamped per
> step). Differences are incidental: head set (`UNION_HEADS` there vs `GOAL_HEADS` here), readout
> (attention-delta/goalscope vs greedy generation), and A3's synthetic `q_reflect` control.


In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")            # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P
P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads model + head masks (once)
from _diff_primitives import *                # helper functions into scope
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP

2026-08-29 15:42:48 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [20]:
# ── multipredicate example: 3 INDEPENDENT predicates, one assignment line each ──
# Each item can be implemented in any order (all orderings correct) — the pivot choice is a
# genuine "which sub-task next", not forced by a data dependency.
# ITEMS = {
#     "a": ("the list of words longer than three characters", "a = [w for w in words if len(w) > 3]"),
#     "b": ("the count of palindromic words",                 "b = sum(w == w[::-1] for w in words)"),
#     "c": ("the length of the longest word",                 "c = max(len(w) for w in words)"),
# }
ITEMS = {
    "a": ("the count of palindromic words",                 "a = sum(w == w[::-1] for w in words)"),
    "b": ("the length of the longest word",                 "b = max(len(w) for w in words)"),
    "c": ("the list of words that starts with a vowel",    "c = [w for w in words if w[0].lower() in 'aeiou']"),
}
ORDER = ["a", "b", "c"]
TASK_SPEC = "\n".join(f"({i+1}) `{k}`: {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill implementing the assignments in `done` (kept in ORDER), pivot at the next indent."""
    lines = ["<code>", "def func(words: list[str]):"]
    for k in ORDER:
        if k in done:
            lines.append(f"    {ITEMS[k][1]}")
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot sits here (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] - 1

# task-item token spans are IDENTICAL across done-sets (task precedes the prefill) — compute once.
_p0 = build_prompt([])
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
show_prompt("done={}      ", _p0, ITEM_SPAN["a"])
show_prompt("done={a}     ", build_prompt(["a"]), ITEM_SPAN["b"])
print("item spans:", ITEM_SPAN, " pivot(∅)=", pivot_token(_p0))

╭─ done={}        TASK span (69, 77)  ·  pivot=['    ']  ·  L=123 ────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) `a`: the count of palindromic words\n                                                                       │
│ (2) `b`: the length of the longest word\n                                                                       │
│ (3) `c`: the list of words that starts with a vowel\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ done={a}       TASK span (84, 90)  ·  pivot=['    ']  ·  L=138 ────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) `a`: the count of palindromic words\n                                                                       │
│ (2) `b`: the length of the longest word\n                                                                       │
│ (3) `c`: the list of words that starts with a vowel\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(w == w[::-1] for w in words)\n                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

item spans: {'a': (69, 77), 'b': (84, 90), 'c': (97, 106)}  pivot(∅)= ['    ']


## A1 · Graded suppression + OV-write norm

Sweep PROGRESS ∈ {∅, {a}, {a,b}}. For each: value-weighted attention from the pivot onto each task
item's spec tokens (mean over GOAL heads), and the norm of the aggregated goal-head write.
**Predict:** attention onto item-k drops sharply when k enters PROGRESS; frontier item retains mass;
write-norm decreases monotonically toward the DIFF==0 (all-done) floor.

In [21]:
DONE_LEVELS = [[], ["a"], ["a", "b"]]
tbl = Table(title="A1 · pivot→TASK-item attention (mean over GOAL heads)  +  ‖goal write‖", highlight=False)
tbl.add_column("PROGRESS")
for k in ORDER: tbl.add_column(f"attn·{k}", justify="right")
tbl.add_column("‖write‖", justify="right")
for done in DONE_LEVELS:
    p = build_prompt(done)
    piv = pivot_of(p)
    tok = prepare_input(prompts=p, tokenizer=mt)
    info = get_attention_matrices(tok, mt, value_weighted=True)
    row = ["{" + ",".join(done) + "}"]
    for k in ORDER:
        a = pivot_attn(info, GOAL_HEADS, list(range(*ITEM_SPAN[k])), piv).sum()
        mark = "[green]" if k not in done else "[red]"
        row.append(f"{mark}{a:.3f}[/]")
    row.append(f"{write_norm(p, GOAL_HEADS, piv):.2f}")
    tbl.add_row(*row)
rich_print(tbl)
rich_print("[grey58]green = still-undone item (expect mass), red = done item (expect suppressed)[/grey58]")

 A1 · pivot→TASK-item attention (mean over GOAL  
             heads)  +  ‖goal write‖             
┏━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ PROGRESS ┃ attn·a ┃ attn·b ┃ attn·c ┃ ‖write‖ ┃
┡━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ {}       │  1.557 │  0.196 │  0.617 │  223.51 │
│ {a}      │  0.203 │  1.237 │  0.521 │  195.83 │
│ {a,b}    │  0.257 │  0.177 │  3.636 │  257.50 │
└──────────┴────────┴────────┴────────┴─────────┘

green = still-undone item (expect mass), red = done item (expect suppressed)

## A2 · Anti-alignment of the query update with the completed item's keys

`q` is pre-RoPE at the pivot; `k` is pre-RoPE at item-a's spec tokens. For every GOAL q-head, cosine of
`Δq = q({a}) − q({})` (q-head slice) against the mean item-a key (its kv-head slice, GQA `h//GROUP`).
**Predict:** cosine < 0 (the completed item is *subtracted into* the query, suppressing its keys).

In [22]:
qheads = heads_per_layer(GOAL_HEADS, "q")
Q_LAYERS = sorted(qheads)
p0, pa = build_prompt([]), build_prompt(["a"])
q0 = cache_src_proj(p0, [pivot_of(p0)], Q_LAYERS, "q")
qa = cache_src_proj(pa, [pivot_of(pa)], Q_LAYERS, "q")
dq = q_lincomb([(1.0, qa), (-1.0, q0)])                       # {layer: (1, width)}

# mean item-a KEY at its spec tokens (positions identical across prompts; use p0)
ka = cache_src_proj(p0, list(range(*ITEM_SPAN["a"])), Q_LAYERS, "k")
kbar = {l: ka[l].mean(0, keepdim=True) for l in Q_LAYERS}     # {layer: (1, width)}

hd = mt.head_dim
coss = []
for (l, h) in GOAL_HEADS:
    if l not in dq: continue
    qv = dq[l][0, h * hd:(h + 1) * hd]
    kvh = h // GROUP
    kv = kbar[l][0, kvh * hd:(kvh + 1) * hd]
    coss.append(cos(qv, kv))
coss = np.array(coss)
print(f"Δq·k(item a) cosine over {len(coss)} GOAL q-heads: "
      f"mean={coss.mean():+.3f}  median={np.median(coss):+.3f}  frac<0={ (coss < 0).mean():.2f}")
print("(strongly negative mean ⇒ completing a subtracts a's key-direction into the query — suppression)")

Δq·k(item a) cosine over 122 GOAL q-heads: mean=-0.060  median=-0.049  frac<0=0.66
(strongly negative mean ⇒ completing a subtracts a's key-direction into the query — suppression)


## A3 · Sign necessity — reflect the done-update

If the *sign* of the query update selects, then reflecting the done-update through baseline —
`q_reflect = q({}) − (q({a,b}) − q({})) = 2·q({}) − q({a,b})` — should **not** behave as if a,b are
done (should not skip to c), unlike the true `q({a,b})`. Injected into the ∅-dest and generated.

In [25]:
# ── show SRC (a,b done) and DEST (∅) prompts + their pivots (see reference cell) ──
pab = build_prompt(["a", "b"])
qab = cache_src_proj(pab, [pivot_of(pab)], Q_LAYERS, "q")
q_reflect = q_lincomb([(2.0, q0), (-1.0, qab)])

def q_inject_specs(qcache, dest_pivot):
    qh = heads_per_layer(GOAL_HEADS, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])

dest_tok = prepare_input(prompts=p0, tokenizer=mt)
piv0 = pivot_of(p0)
pf_ab, dc_ab = q_inject_specs(qab, piv0)
pf_rf, dc_rf = q_inject_specs(q_reflect, piv0)

show_prompt("SRC  pab = {a,b} done", pab, ITEM_SPAN["c"])   # pivot shown in panel title
show_prompt("DEST p0  = {}  (∅)   ", p0,  ITEM_SPAN["a"])

rich_print(f"[grey70]clean ∅ (expect writes a) :[/grey70] {escape(repr(greedy(dest_tok, max_new=32)))}")
rich_print(f"[sea_green3]inject q(a,b) (expect c)  :[/sea_green3] "
           f"{escape(repr(greedy(dest_tok, prefill_specs=pf_ab, decode_specs=dc_ab, n_steer=40, max_new=64)))}")
rich_print(f"[salmon1]inject q_reflect (expect ≠c):[/salmon1] "
           f"{escape(repr(greedy(dest_tok, prefill_specs=pf_rf, decode_specs=dc_rf, n_steer=40, max_new=64)))}")

╭─ SRC  pab = {a,b} done  TASK span (97, 106)  ·  pivot=['    ']  ·  L=150 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) `a`: the count of palindromic words\n                                                                       │
│ (2) `b`: the length of the longest word\n                                                                       │
│ (3) `c`: the list of words that starts with a vowel\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(w == w[::-1] for w in words)\n                                                                      │
│     b = max(len(w) for w in words)\n                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DEST p0  = {}  (∅)     TASK span (69, 77)  ·  pivot=['    ']  ·  L=123 ────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) `a`: the count of palindromic words\n                                                                       │
│ (2) `b`: the length of the longest word\n                                                                       │
│ (3) `c`: the list of words that starts with a vowel\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean ∅ (expect writes a) : ' a = sum(1 for word in words if word == word[::-1])\n     b = max(len(word) for word 
in words) if words else '

inject q(a,b) (expect c)  : " vowels = {'a', 'e', 'i', 'o', 'u'}\n     a = [word for word in words if 
word[0].lower() in vowels]\n     b = [word for word in words if word == word[::-1]]\n     c = max(len(word) for 
word in words)"

inject q_reflect (expect ≠c): " a = sum(1 for word in words if word == word[::-1])\n     b = sum(1 for word in 
words if word == word[::-1])\n     c = [word for word in words if word[0].lower() in 'aeiou']\n     return a, 
max(len(word) for word"

## B1 · Query linearity (geometric)

Is the done-set encoded additively in query space? Cosine, over the concatenated GOAL q-head slices,
between `Δq({a,b})` and `Δq({a}) + Δq({b})`  (each `Δq(·) = q(·) − q({})`).
**Predict:** high cosine ⇒ subtraction is linear/compositional over done-items.

In [29]:
pb = build_prompt(["b"])
qb = cache_src_proj(pb, [pivot_of(pb)], Q_LAYERS, "q")

def goal_q_vec(qcache):
    parts = [qcache[l][0, h * hd:(h + 1) * hd] for (l, h) in GOAL_HEADS if l in qcache]
    return torch.cat(parts)

dq_a  = goal_q_vec(q_lincomb([(1.0, qa),  (-1.0, q0)]))
dq_b  = goal_q_vec(q_lincomb([(1.0, qb),  (-1.0, q0)]))
dq_ab = goal_q_vec(q_lincomb([(1.0, qab), (-1.0, q0)]))
print(f"cos( Δq(a,b) , Δq(a)+Δq(b) )      = {cos(dq_ab, dq_a + dq_b):+.3f}")
print(f"cos( Δq(a,b) , Δq(a) )            = {cos(dq_ab, dq_a):+.3f}   (single-term baseline)")
print(f"cos( Δq(a)   , Δq(b) )            = {cos(dq_a,  dq_b):+.3f}   (are the two updates ~orthogonal?)")

cos( Δq(a,b) , Δq(a)+Δq(b) )      = +0.801
cos( Δq(a,b) , Δq(a) )            = +0.537   (single-term baseline)
cos( Δq(a)   , Δq(b) )            = +0.474   (are the two updates ~orthogonal?)


## B2 · Compositional causal injection  (the payload)

Inject a *synthesised* query `q_synth = q({a}) + q({b}) − q({})` at the ∅-dest pivot and generate.
**Predict:** writes item **c** only — the composed done-set behaves as {a,b}. Controls: real `q({a,b})`
should match; single terms narrow less. **Saturation** `q(a)+q(b)+q(c)−2·q({})` → all done → **boundary/return**.

*Note:* injection target is the fixed-geometry ∅ prompt; only the position-free pre-RoPE queries are combined.

In [30]:
pc = build_prompt(["c"])
qc = cache_src_proj(pc, [pivot_of(pc)], Q_LAYERS, "q")
q_synth = q_lincomb([(1.0, qa), (1.0, qb), (-1.0, q0)])                       # ≈ done {a,b} -> expect c
q_sat   = q_lincomb([(1.0, qa), (1.0, qb), (1.0, qc), (-2.0, q0)])            # done {a,b,c} -> expect boundary

def gen_inject(qcache, tag, color):
    pf, dc = q_inject_specs(qcache, piv0)
    rich_print(f"[{color}]{tag}:[/{color}] "
               f"{escape(repr(greedy(dest_tok, prefill_specs=pf, decode_specs=dc, n_steer=32, max_new=64)))}")

rich_print(f"[grey70]clean ∅            (expect a) :[/grey70] {escape(repr(greedy(dest_tok, max_new=64)))}")
gen_inject(qab,     "inject q(a,b) real (expect c) ", "sea_green3")
gen_inject(q_synth, "inject q_synth=a+b-∅ (expect c)", "gold3")
gen_inject(qa,      "inject q({a})       (expect b/c)", "grey58")
gen_inject(q_sat,   "inject q_sat=a+b+c-2∅ (→bound)", "salmon1")

clean ∅            (expect a) : " a = sum(1 for word in words if word == word[::-1])\n     b = max(len(word) for 
word in words) if words else 0\n     c = [word for word in words if word[0].lower() in 'aeiou']\n     return a, b, 
c\n</"

inject q(a,b) real (expect c) : " vowels = {'a', 'e', 'i', 'o', 'u'}\n     a = [word for word in words if 
word[0].lower() in vowels]\n     b = max(len(word) for word in words)\n     c = sum(1 for word in words if word == 
word[::-1"

inject q_synth=a+b-∅ (expect c): ' a = 0\n     b = 0\n     c = []\n     vowels = set(&#39;aeiou&#39;)\n     for 
word in words:\n         if word == word[::-1]:\n             a += 1\n         if len(word) > b:\n             b = 
len(word'

inject q({a})       (expect b/c): " a = max((len(word) for word in words), default=0)\n     b = [word for word in 
words if len(word) == a]\n     c = sum(1 for word in words if word.lower().startswith(('a', 'e', 'i', 'o', 'u')))\n
d"

inject q_sat=a+b+c-2∅ (→bound): ' a = max(len(word) for word in words) if words else 0\n     b = [word for word in 
words if len(word) == a][0] if words else ""\n     c = sum(1 for word in words if word.lower().startswith((\'a\', 
\'e\', \'i\', \''

## B2b · De-overlap correction (Gram–Schmidt)  —  is the shared component double-counted?

`Δa = q({a})−q0` and `Δb = q({b})−q0` are **not orthogonal** (`cos(Δa,Δb) ≠ 0`), so the plain synth
`q0 + Δa + Δb` counts their shared component `s` **twice**. Since the real `Δab = q({a,b})−q0` is
**already cached**, we can settle it directly — no theory:

- **add:** `Δa + Δb`  (current B2)
- **GS :** `Δa + Δb − projₐ(Δb)`  — subtract the *shared* part (the **projection**, computed per layer), keep `−q0`.

Compare each reconstruction's `cos(·, Δab)` and residual. **If `add` already tracks `Δab`** (strong B1
additivity) the model double-counts nothing → GS would *over-correct*. **If `GS` is closer**, the
overlap is real and the GS-corrected query is the better payload. Then inject all three and read behavior.

In [ ]:
# ── B2b: measure which reconstruction matches the real Δab, then inject the GS-corrected query ──
import torch
import torch.nn.functional as F

def _flat(d): return torch.cat([d[l].flatten() for l in Q_LAYERS])
Delta_ab = _flat(qab) - _flat(q0)                       # ground-truth "both done" update

add_parts, gs_parts, cos_ab = [], [], []
q_synth_gs = {}                                          # {layer: (1, width)} for injection
for l in Q_LAYERS:
    da = (qa[l] - q0[l])                                 # (1, width)
    db = (qb[l] - q0[l])
    fa, fb = da.flatten(), db.flatten()
    proj = (fa @ fb) / (fa @ fa) * da                    # shared component of Δb along Δa (per layer)
    add_parts.append((da + db).flatten())
    gs_parts.append((da + db - proj).flatten())
    cos_ab.append((fa @ fb) / (fa.norm() * fb.norm()))
    q_synth_gs[l] = q0[l] + da + db - proj               # = q0 + Δa + Δb_perp   (keeps −q0)

add_recon = torch.cat(add_parts)
gs_recon  = torch.cat(gs_parts)
rich_print(f"[grey70]mean per-layer cos(Δa, Δb) = {torch.stack(cos_ab).mean():.3f}   "
           f"(0 ⇒ orthogonal, no double-count)[/grey70]")
for name, v in [("add  Δa+Δb", add_recon), ("GS   Δa+Δb−projₐΔb", gs_recon)]:
    rich_print(f"  {name:22s}  cos(·,Δab)={F.cosine_similarity(v, Delta_ab, dim=0).item():+.3f}   "
               f"‖resid‖={(v - Delta_ab).norm():.2f}")

# ── behavioral read: inject add / GS / real, all via the same machinery (gen_inject from B2) ──
rich_print(f"[grey70]clean ∅                    (expect a) :[/grey70] {escape(repr(greedy(dest_tok, max_new=64)))}")
gen_inject(q_synth,    "inject q_synth   = a+b−∅            (expect c)", "gold3")
gen_inject(q_synth_gs, "inject q_synth_gs = a+b−∅−projₐΔb   (expect c)", "cyan")
gen_inject(qab,        "inject q(a,b) real                 (expect c)", "sea_green3")


mean per-layer cos(Δa, Δb) = 0.622   (0 ⇒ orthogonal, no double-count)

add  Δa+Δb              cos(·,Δab)=+0.837   ‖resid‖=313.33

GS   Δa+Δb−projₐΔb      cos(·,Δab)=+0.853   ‖resid‖=197.99

clean ∅                    (expect a) : " a = sum(1 for word in words if word == word[::-1])\n     b = 
max(len(word) for word in words) if words else 0\n     c = [word for word in words if word[0].lower() in 'aeiou']\n
return a, b, c\n</"

inject q_synth   = a+b−∅            (expect c): ' a = 0\n     b = 0\n     c = []\n     vowels = 
set(&#39;aeiou&#39;)\n     for word in words:\n         if word == word[::-1]:\n             a += 1\n         if 
len(word) > b:\n             b = len(word'

inject q_synth_gs = a+b−∅−projₐΔb   (expect c): ' vowels = set(&#39;aeiouAEIOU&#39;)\n     a = 0\n     b = 0\n     
c = []\n     for word in words:\n         if word == word[::-1]:\n             a += 1\n         if len(word) > b:\n
b'

inject q(a,b) real                 (expect c): " vowels = {'a', 'e', 'i', 'o', 'u'}\n     a = [word for word in 
words if word[0].lower() in vowels]\n     b = max(len(word) for word in words)\n     c = sum(1 for word in words if
word == word[::-1"